In [1]:
import importlib.util
import numpy as np
import torch
from pathlib import Path
from Bio.PDB import PDBParser

MODEL = "11_30042026"

_spec = importlib.util.spec_from_file_location('pocket_model', f'./models/{MODEL}.py')
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

PocketFinder                = _mod.PocketFinder
AA_TO_IDX                   = _mod.AA_TO_IDX
PHYSCHEM                    = _mod.PHYSCHEM
compute_structural_features = _mod.compute_structural_features
parse_mol2_protein          = _mod.parse_mol2_protein

In [2]:
CHECKPOINT = f'./models/{MODEL}.pt'

state    = torch.load(CHECKPOINT, map_location='cpu')
in_dim   = state['embedding.weight'].shape[1]
hidden   = state['embedding.weight'].shape[0]
n_layers = sum(1 for k in state if k.startswith('convs.') and k.endswith('.eps'))

model = PocketFinder(in_dim=in_dim, hidden=hidden, n_layers=n_layers)
model.load_state_dict(state)
model.eval()
print(f'loaded: in_dim={in_dim}  hidden={hidden}  n_layers={n_layers}  params={sum(p.numel() for p in model.parameters())}')

loaded: in_dim=15  hidden=256  n_layers=4  params=540737


In [3]:
_pdb_parser = PDBParser(QUIET=True)


def parse_pdb_protein(path, chain_id=None):
    structure = _pdb_parser.get_structure('p', str(path))
    coords, residue_names, res_ids = [], [], []
    for residue in structure.get_residues():
        if residue.get_id()[0] != ' ':
            continue
        chain = residue.get_parent().get_id()
        if chain_id is not None and chain != chain_id:
            continue
        res_name = residue.get_resname()
        if res_name not in AA_TO_IDX or 'CA' not in residue:
            continue
        coords.append(residue['CA'].get_vector().get_array())
        residue_names.append(res_name)
        res_ids.append(f"{chain}:{res_name}{residue.get_id()[1]}")

    coords     = np.array(coords, dtype=np.float32)
    physchem   = np.array([PHYSCHEM[r] for r in residue_names], dtype=np.float32)
    structural = compute_structural_features(coords)

    def norm(x):
        std = x.std(axis=0)
        std[std < 1e-8] = 1.0
        return (x - x.mean(axis=0)) / std

    h = np.concatenate([norm(physchem), norm(structural)], axis=1)
    pos = torch.tensor(coords, dtype=torch.float32)
    h   = torch.tensor(h, dtype=torch.float32)
    return pos, h, res_ids


def load_protein(path, chain_id=None):
    path = Path(path)
    if path.suffix == '.pdb':
        return parse_pdb_protein(path, chain_id=chain_id)
    elif path.suffix in ('.mol2', '.mol'):
        pos, h = parse_mol2_protein(path)
        res_ids = [str(i) for i in range(len(pos))]
        return pos, h, res_ids
    else:
        raise ValueError(f'unsupported format: {path.suffix}')

In [4]:
def predict_pocket(protein_path, threshold=0.5, chain_id=None):
    pos, h, res_ids = load_protein(protein_path, chain_id=chain_id)
    batch = torch.zeros(len(pos), dtype=torch.long)

    with torch.no_grad():
        probs = model(h, pos, batch).sigmoid()

    pocket_mask    = probs > threshold
    pocket_indices = pocket_mask.nonzero(as_tuple=True)[0].tolist()
    pocket_res_ids = [res_ids[i] for i in pocket_indices]
    pocket_coords  = pos[pocket_mask]

    return {
        'probs':          probs,
        'pocket_indices': pocket_indices,
        'pocket_res_ids': pocket_res_ids,
        'pocket_coords':  pocket_coords,
        'all_coords':     pos,
        'res_ids':        res_ids,
    }

In [5]:
PROTEIN = "1a2b_1"
PROTEIN_PATH = f'./data/scPDB/{PROTEIN}/protein.mol2'
SITE_PATH    = f'./data/scPDB/{PROTEIN}/site.mol2'

result = predict_pocket(PROTEIN_PATH, threshold=0.5)

print(f"total residues : {len(result['res_ids'])}")
print(f"pocket residues: {len(result['pocket_indices'])} ({100*len(result['pocket_indices'])/len(result['res_ids']):.1f}%)")
print(f"prob range     : [{result['probs'].min().item():.3f}, {result['probs'].max().item():.3f}]")

total residues : 178
pocket residues: 42 (23.6%)
prob range     : [0.000, 1.000]


In [6]:
DIST_THRESHOLD = 1.0  # same as training labels

site_coords = _mod.parse_mol2_coords(SITE_PATH)
labels = (torch.cdist(result['all_coords'], site_coords).min(dim=1).values < DIST_THRESHOLD).float()

probs = result['probs']
preds = (probs > 0.5).float()

tp = (preds * labels).sum()
fp = (preds * (1 - labels)).sum()
fn = ((1 - preds) * labels).sum()
tn = ((1 - preds) * (1 - labels)).sum()

precision = tp / (tp + fp + 1e-8)
recall    = tp / (tp + fn + 1e-8)
f1        = 2 * precision * recall / (precision + recall + 1e-8)

print(f"ground-truth pocket residues : {labels.sum().int().item()} / {len(labels)}")
print(f"predicted pocket residues    : {preds.sum().int().item()} / {len(preds)}")
print()
print(f"precision : {precision:.3f}")
print(f"recall    : {recall:.3f}")
print(f"F1        : {f1:.3f}")
print()
print(f"TP={tp.int().item()}  FP={fp.int().item()}  FN={fn.int().item()}  TN={tn.int().item()}")

ground-truth pocket residues : 34 / 178
predicted pocket residues    : 42 / 178

precision : 0.810
recall    : 1.000
F1        : 0.895

TP=34  FP=8  FN=0  TN=136


In [7]:
# top-k residues by probability regardless of threshold
K = 20
top_k = result['probs'].topk(K)
print(f'top-{K} pocket residues:')
for prob, idx in zip(top_k.values.tolist(), top_k.indices.tolist()):
    print(f'  {result["res_ids"][idx]:>12s}  p={prob:.3f}')

top-20 pocket residues:
           156  p=1.000
           155  p=0.998
           114  p=0.995
           157  p=0.994
           113  p=0.992
           115  p=0.992
           116  p=0.992
           158  p=0.991
            28  p=0.989
            30  p=0.988
            27  p=0.984
            31  p=0.982
            57  p=0.982
           159  p=0.978
            29  p=0.975
            26  p=0.974
            15  p=0.971
            16  p=0.970
            33  p=0.967
            17  p=0.959


In [8]:
from scipy.spatial import cKDTree

def pocket_to_grid(pocket_coords, protein_coords, spacing=1.0, padding=6.0, max_dist_from_pocket=6.0):
    bb_min = pocket_coords.min(0) - padding
    bb_max = pocket_coords.max(0) + padding
    grid_shape = np.ceil((bb_max - bb_min) / spacing).astype(int)

    xi = np.arange(grid_shape[0]) * spacing + bb_min[0]
    yi = np.arange(grid_shape[1]) * spacing + bb_min[1]
    zi = np.arange(grid_shape[2]) * spacing + bb_min[2]
    grid = np.stack(np.meshgrid(xi, yi, zi, indexing='ij'), axis=-1).reshape(-1, 3)

    # remove points inside protein atoms first
    PROTEIN_OCCUPANCY = 1.4
    protein_tree    = cKDTree(protein_coords)
    dist_protein, _ = protein_tree.query(grid)
    grid            = grid[dist_protein >= PROTEIN_OCCUPANCY]

    # then keep only points near pocket atoms
    pocket_tree    = cKDTree(pocket_coords)
    dist_pocket, _ = pocket_tree.query(grid)
    return grid[dist_pocket < max_dist_from_pocket]


pocket_coords  = result['pocket_coords'].numpy()
protein_coords = result['all_coords'].numpy()

grid = pocket_to_grid(pocket_coords, protein_coords)
print(f'grid points: {len(grid)}')

grid points: 12781


In [9]:
import pyrite
from pyrite import Mol, Viewer
from pyrite.bounds import Pocket

pocket = Pocket(grid, np.ones_like(grid[:, 0]) * 0.7)

receptor = pyrite.Mol.from_pdb(PROTEIN_PATH)



ArgumentError: Python argument types in
    rdkit.Chem.rdmolops.SanitizeMol(Mol)
did not match C++ signature:
    SanitizeMol(RDKit::ROMol {lvalue} mol, unsigned long long sanitizeOps=rdkit.Chem.rdmolops.SanitizeFlags.SANITIZE_ALL, bool catchErrors=False)